In [0]:
from pyspark.sql import DataFrame, functions as F
from pyspark.sql.window import Window


bronze_table = "de_e2e.bronze.douyin_api_raw"
silver_table = "de_e2e.silver.aweme_snapshot_delta"

bucket = "de-e2e-413612133697-ap-southeast-1-an"
silver_path = f"s3://{bucket}/lakehouse/silver/douyin/aweme_snapshot_delta/"

In [0]:
bucket = "de-e2e-413612133697-ap-southeast-1-an"

spark.sql("DROP TABLE IF EXISTS de_e2e.silver.douyin_aweme_media")
spark.sql("DROP TABLE IF EXISTS de_e2e.silver.douyin_aweme_snapshot")

dbutils.fs.rm(f"s3://{bucket}/lakehouse/silver/douyin/douyin_aweme_media/", True)
dbutils.fs.rm(f"s3://{bucket}/lakehouse/silver/douyin/aweme_snapshot_delta/", True)

In [0]:
bronze_df = spark.table(bronze_table)
#Current struct in S3: raw_struct.raw.raw.aweme_list
def with_aweme_array(df: DataFrame) -> DataFrame:
    return df.withColumn(
        "aweme_list",
        F.col("raw_struct.raw.raw.aweme_list"),
    )

snapshot_df = (
    bronze_df
    .transform(with_aweme_array)
    .withColumn("aweme", F.explode_outer("aweme_list"))
    .select(
        F.col("niche"),
        F.col("account_id"),
        F.col("account_type"),
        F.col("source_file"),
        F.col("generated_at").alias("landing_generated_at"),
        F.to_date(F.col("generated_at")).alias("snapshot_date"),
        F.col("bronze_ingested_at"),
        F.col("media_manifest_uri"),

        F.col("aweme.aweme_id").cast("string").alias("aweme_id"),
        F.col("aweme.group_id").cast("string").alias("group_id"),
        F.col("aweme.sec_item_id").cast("string").alias("sec_item_id"),
        F.col("aweme.share_url").alias("share_url"),

        F.col("aweme.desc").alias("description"),
        F.col("aweme.item_title").alias("item_title"),
        F.col("aweme.video_text").alias("video_text"),

        F.col("aweme.create_time").cast("long").alias("create_time_epoch"),
        F.from_unixtime(F.col("aweme.create_time").cast("long")).cast("timestamp").alias("created_at"),

        F.col("aweme.duration").cast("long").alias("duration_ms"),
        (F.col("aweme.duration").cast("double") / F.lit(1000.0)).alias("duration_seconds"),
        F.col("aweme.media_type").cast("int").alias("media_type"),
        F.col("aweme.aweme_type").cast("int").alias("aweme_type"),
        F.col("aweme.region").alias("region"),
        F.col("aweme.is_ads").cast("boolean").alias("is_ads"),
        F.col("aweme.is_top").cast("int").alias("is_top"),
        F.col("aweme.prevent_download").cast("boolean").alias("prevent_download"),

        F.col("aweme.author.uid").cast("string").alias("author_uid"),
        F.col("aweme.author.sec_uid").cast("string").alias("author_sec_uid"),
        F.col("aweme.author.nickname").alias("author_nickname"),
        F.col("aweme.author.signature_extra").alias("author_signature"),
        F.col("aweme.author.custom_verify").alias("author_custom_verify"),
        F.col("aweme.author.enterprise_verify_reason").alias("author_enterprise_verify_reason"),

        F.col("aweme.music.id_str").cast("string").alias("music_id"),
        F.col("aweme.music.title").alias("music_title"),
        F.col("aweme.music.author").alias("music_author"),
        F.col("aweme.music.is_original").cast("boolean").alias("music_is_original"),

        F.col("aweme.statistics.digg_count").cast("long").alias("like_count"),
        F.col("aweme.statistics.comment_count").cast("long").alias("comment_count"),
        F.col("aweme.statistics.share_count").cast("long").alias("share_count"),
        F.col("aweme.statistics.collect_count").cast("long").alias("collect_count"),
        F.col("aweme.statistics.recommend_count").cast("long").alias("recommend_count"),
        F.col("aweme.statistics.play_count").cast("long").alias("play_count"),
        F.col("aweme.statistics.admire_count").cast("long").alias("admire_count"),

        F.col("aweme.text_extra").alias("hashtags_raw"),
        F.col("aweme.anchors").alias("anchors_raw"),
        F.col("aweme.interest_points").alias("interest_points_raw"),
        F.col("aweme.position").alias("position_raw"),
        F.col("aweme.video").alias("video_raw"),
        F.col("aweme.images").alias("images_raw"),
        F.col("aweme").alias("raw_aweme"),

        F.current_timestamp().alias("silver_ingested_at"),
    )
    .where(F.col("aweme_id").isNotNull())
)

display(
    snapshot_df.select(
        "account_id",
        "aweme_id",
        "source_file",
        "snapshot_date",
        "landing_generated_at",
        "description",
        "like_count",
        "comment_count",
        "share_count",
        "collect_count",
        "recommend_count",
        "play_count",
    ).orderBy("account_id", "aweme_id", "landing_generated_at").limit(100)
)



In [0]:
from delta.tables import DeltaTable
if DeltaTable.isDeltaTable(spark, silver_path):
    target = DeltaTable.forPath(spark, silver_path)
    (
        target.alias("t")
        .merge(
            snapshot_df.alias("s"),
            """
            t.account_id = s.account_id
            AND t.aweme_id = s.aweme_id
            AND t.source_file = s.source_file
            """,
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )
else:
    (
        snapshot_df.write
        .format("delta")
        .mode("overwrite")
        .save(silver_path)
    )

spark.sql(
    f"""
    CREATE TABLE IF NOT EXISTS {silver_table}
    USING DELTA
    LOCATION '{silver_path}'
    """
)

In [0]:
result_df = spark.table(silver_table)

print(f"Silver aweme snapshot rows: {result_df.count()}")

display(
    result_df.groupBy("snapshot_date")
    .agg(
        F.count("*").alias("snapshot_rows"),
        F.countDistinct("aweme_id").alias("aweme_count"),
        F.countDistinct("account_id").alias("creator_count"),
    )
    .orderBy(F.col("snapshot_date").desc())
)